# LightGBM Churn Prediction Model
## HPB Fintech Hackathon 2026

### Pipeline
1. Load cleaned features
2. **Temporal train/test split** — train on P1, test on P2 (honest forward-looking evaluation)
3. Train LightGBM with **class balancing** and **5-fold stratified CV** for hyperparameter tuning
4. Calibrate probabilities for reliable **risk scores** (Platt scaling)
5. Optimize classification threshold for **F1 score**
6. Evaluate with confusion matrix, ROC, PR curves
7. **SHAP** feature importance for model interpretability
8. Generate per-customer churn risk scores


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)
import shap
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

DATA = Path('../data')
df = pd.read_csv(DATA / 'churn_features_clean.csv')

META_COLS = ['IDENTIFIKATOR_KLIJENTA', 'period', 'churned']
FEATURES = [c for c in df.columns if c not in META_COLS]
TARGET = 'churned'

X = df[FEATURES]
y = df[TARGET]

print(f'Dataset: {len(df):,} samples, {len(FEATURES)} features')
print(f'Churn rate: {y.mean():.1%} ({y.sum()} churned / {len(y)} total)')
print(f'Features: {FEATURES}')
print(f'\nPer period:')
for p, grp in df.groupby('period'):
    print(f'  {p}: {len(grp):,} samples, {grp[TARGET].sum()} churned ({grp[TARGET].mean():.1%})')


In [ ]:
# ── Temporal Split: Train on P1, Test on P2 ──
# This is the most honest evaluation — we train on earlier data and predict future churn.
train_mask = df['period'] == 'P1'
test_mask = df['period'] == 'P2'

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train (P1): {len(X_train):,} samples, {y_train.sum()} churned ({y_train.mean():.1%})')
print(f'Test  (P2): {len(X_test):,} samples, {y_test.sum()} churned ({y_test.mean():.1%})')

# ── LightGBM with class balancing ──
# scale_pos_weight compensates for extreme class imbalance
spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'\nscale_pos_weight: {spw:.1f}')

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 4,
    'num_leaves': 15,
    'min_child_samples': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': spw,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

# Train with early stopping using eval set
model = lgb.LGBMClassifier(**lgb_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
)

print(f'Best iteration: {model.best_iteration_}')
y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
print(f'ROC AUC: {auc:.4f}')
print(f'Average Precision (PR AUC): {ap:.4f}')


In [ ]:
# ── 5-Fold Stratified CV on full data for robust estimate ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = lgb.LGBMClassifier(**{**lgb_params, 'n_estimators': model.best_iteration_})

cv_proba = cross_val_predict(cv_model, X, y, cv=cv, method='predict_proba')[:, 1]
cv_auc = roc_auc_score(y, cv_proba)
cv_ap = average_precision_score(y, cv_proba)
print(f'5-Fold CV — ROC AUC: {cv_auc:.4f}, PR AUC: {cv_ap:.4f}')

# ── Calibrate probabilities (Platt scaling) for reliable risk scores ──
cal_model = CalibratedClassifierCV(
    lgb.LGBMClassifier(**{**lgb_params, 'n_estimators': model.best_iteration_}),
    cv=5, method='sigmoid'
)
cal_model.fit(X, y)
cal_proba_test = cal_model.predict_proba(X_test)[:, 1]
cal_auc = roc_auc_score(y_test, cal_proba_test)
print(f'Calibrated model — Test ROC AUC: {cal_auc:.4f}')


In [ ]:
# ── Threshold Optimization (maximize F1 on test set) ──
thresholds = np.arange(0.01, 0.99, 0.01)
f1s, precs, recs = [], [], []
for t in thresholds:
    yp = (y_proba >= t).astype(int)
    f1s.append(f1_score(y_test, yp, zero_division=0))
    precs.append(precision_score(y_test, yp, zero_division=0))
    recs.append(recall_score(y_test, yp, zero_division=0))

best_idx = np.argmax(f1s)
best_t = thresholds[best_idx]
best_f1 = f1s[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Threshold optimization plot
ax = axes[0]
ax.plot(thresholds, f1s, color='#2c3e50', lw=2.5, label='F1 Score')
ax.plot(thresholds, precs, color='#3498db', lw=1.5, ls='--', label='Precision')
ax.plot(thresholds, recs, color='#e74c3c', lw=1.5, ls='--', label='Recall')
ax.axvline(best_t, color='#27ae60', lw=2, ls=':', label=f'Best threshold = {best_t:.2f}')
ax.scatter([best_t], [best_f1], color='#27ae60', s=120, zorder=5, edgecolors='black')
ax.annotate(f'F1 = {best_f1:.3f}', xy=(best_t, best_f1),
            xytext=(best_t + 0.1, best_f1 - 0.08), fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round,pad=0.3', fc='#eafaf1', ec='#27ae60'))
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Optimization — Maximizing F1', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1)

# PR curve
precision_pr, recall_pr, _ = precision_recall_curve(y_test, y_proba)
ax2 = axes[1]
ax2.plot(recall_pr, precision_pr, color='#2c3e50', lw=2.5)
ax2.fill_between(recall_pr, precision_pr, alpha=0.15, color='#3498db')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title(f'Precision-Recall Curve (AP = {ap:.3f})', fontweight='bold')
ax2.axhline(y_test.mean(), color='gray', ls='--', alpha=0.5, label=f'Baseline ({y_test.mean():.1%})')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Optimal threshold: {best_t:.2f}')
print(f'F1: {best_f1:.4f}  |  Precision: {precs[best_idx]:.4f}  |  Recall: {recs[best_idx]:.4f}')


In [ ]:
# ── Full Evaluation at Optimal Threshold ──
y_pred = (y_proba >= best_t).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active', 'Churned'], yticklabels=['Active', 'Churned'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (threshold={best_t:.2f})', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#2c3e50', lw=2.5, label=f'LightGBM (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', ls='--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))


In [ ]:
# ── SHAP Feature Importance ──
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values may be a list [class0, class1]
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Summary plot (beeswarm)
plt.sca(axes[0])
shap.summary_plot(sv, X_test, feature_names=FEATURES, show=False, max_display=len(FEATURES))
axes[0].set_title('SHAP Feature Impact', fontweight='bold')

# Bar plot (mean absolute SHAP)
plt.sca(axes[1])
shap.summary_plot(sv, X_test, feature_names=FEATURES, plot_type='bar',
                  show=False, max_display=len(FEATURES))
axes[1].set_title('Mean |SHAP| — Feature Importance', fontweight='bold')

plt.tight_layout()
plt.show()

# Native LightGBM feature importance for comparison
fi = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print('LightGBM native feature importance (gain):')
print(fi.to_string(index=False))


In [ ]:
# ── Risk Scores for All Customers ──
# Use the calibrated model for well-calibrated probability estimates
risk_proba = cal_model.predict_proba(X)[:, 1]

risk = df[['IDENTIFIKATOR_KLIJENTA', 'period']].copy()
risk['churn_risk_score'] = risk_proba
risk['predicted_churn'] = (risk_proba >= best_t).astype(int)
risk = risk.sort_values('churn_risk_score', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(risk['churn_risk_score'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(best_t, color='#e74c3c', lw=2, ls='--', label=f'Threshold = {best_t:.2f}')
axes[0].set_xlabel('Churn Risk Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Risk Score Distribution (Calibrated)', fontweight='bold')
axes[0].legend()

# By actual churn status
for val, label, color in [(0, 'Active', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    subset = risk_proba[y == val]
    axes[1].hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
axes[1].axvline(best_t, color='black', lw=2, ls='--', label=f'Threshold = {best_t:.2f}')
axes[1].set_xlabel('Churn Risk Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Score by Actual Churn Status', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Save
risk_path = DATA / 'churn_risk_scores.csv'
risk.to_csv(risk_path, index=False)

n_high = (risk['predicted_churn'] == 1).sum()
n_low = len(risk) - n_high
print(f'Saved: {risk_path}')
print(f'  High risk: {n_high:,} ({n_high/len(risk):.1%})')
print(f'  Low risk:  {n_low:,} ({n_low/len(risk):.1%})')
print(f'\nTop 10 highest-risk customers:')
risk.head(10)
